# Notebook 10 — Continuous Batching and Paged KV Cache

This notebook explains why vLLM and similar runtimes can keep the accelerator busy under bursty traffic. The key ideas are continuous batching at the scheduler and a managed KV-cache allocator under the hood.

## What you will build

- a mixed request stream with staggered arrivals
- a comparison between windowed batching and continuous batching
- a token-level schedule trace for active sequences over time
- a paged or managed KV-cache simulation that highlights fragmentation
- an operating brief that turns the notebook into deployment intuition

## Terminology note

Older explanations often say only `PagedAttention`. Current practice is broader: people also talk about a managed KV cache, block tables, or paged allocators. The common idea is that sequence state is stored in reusable blocks instead of one fragile contiguous slab per request.

In [ ]:
# --- Systems Notebook Setup ---
!pip install -q "transformers>=4.51.0" accelerate torch numpy pandas matplotlib fastapi uvicorn pydantic httpx psutil

import asyncio
import json
import math
import random
import statistics
import threading
import time
from collections import Counter, defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-systems")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def now_ms():
    return time.perf_counter() * 1000

def summarize(values):
    return {
        "count": len(values),
        "mean": statistics.mean(values) if values else 0.0,
        "median": statistics.median(values) if values else 0.0,
        "min": min(values) if values else 0.0,
        "max": max(values) if values else 0.0,
    }

print("✓ Systems notebook environment ready")
print("  Cache dir:", CACHE_DIR)
print("  Artifact root:", ARTIFACT_ROOT)

## Step 1: Add helpers and an artifact path

We will save schedule traces, allocator summaries, and a short operating brief for later notebooks.

In [ ]:
random.seed(10)

ARTIFACT_DIR = ARTIFACT_ROOT / "10_continuous_batching_and_paged_kv_cache"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def write_json(path, payload):
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")

def percentile(values, pct):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return 0.0
    rank = (len(ordered) - 1) * (pct / 100)
    lower = math.floor(rank)
    upper = math.ceil(rank)
    if lower == upper:
        return ordered[int(rank)]
    weight = rank - lower
    return ordered[lower] * (1 - weight) + ordered[upper] * weight

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Step 2: Define a bursty request stream

The arrival pattern mixes short classification jobs with larger chat and retrieval-heavy prompts. That is where continuous batching starts to matter.

In [ ]:
request_stream = [
    {"request_id": "r01", "arrival_ms": 0, "prompt_tokens": 420, "output_tokens": 140, "tenant": "mentor"},
    {"request_id": "r02", "arrival_ms": 5, "prompt_tokens": 1100, "output_tokens": 220, "tenant": "mentor"},
    {"request_id": "r03", "arrival_ms": 11, "prompt_tokens": 260, "output_tokens": 64, "tenant": "extract"},
    {"request_id": "r04", "arrival_ms": 17, "prompt_tokens": 780, "output_tokens": 160, "tenant": "mentor"},
    {"request_id": "r05", "arrival_ms": 26, "prompt_tokens": 320, "output_tokens": 72, "tenant": "ops"},
    {"request_id": "r06", "arrival_ms": 33, "prompt_tokens": 1450, "output_tokens": 260, "tenant": "research"},
    {"request_id": "r07", "arrival_ms": 39, "prompt_tokens": 560, "output_tokens": 110, "tenant": "mentor"},
    {"request_id": "r08", "arrival_ms": 43, "prompt_tokens": 240, "output_tokens": 48, "tenant": "extract"},
    {"request_id": "r09", "arrival_ms": 51, "prompt_tokens": 980, "output_tokens": 180, "tenant": "research"},
    {"request_id": "r10", "arrival_ms": 62, "prompt_tokens": 390, "output_tokens": 86, "tenant": "ops"},
    {"request_id": "r11", "arrival_ms": 69, "prompt_tokens": 660, "output_tokens": 128, "tenant": "mentor"},
    {"request_id": "r12", "arrival_ms": 74, "prompt_tokens": 280, "output_tokens": 54, "tenant": "extract"},
]

stream_df = pd.DataFrame(request_stream)
print(stream_df.to_markdown(index=False))

## Step 3: Visualize the arrival pattern

The chart below shows why fixed request groups are awkward: the work does not arrive on a neat cadence and the prompt sizes are not uniform.

In [ ]:
ax = stream_df.plot.scatter(x="arrival_ms", y="prompt_tokens", s=stream_df["output_tokens"] * 3, figsize=(8, 4), color="#4c78a8", title="Request arrivals and prompt sizes")
ax.set_xlabel("arrival time (ms)")
ax.set_ylabel("prompt tokens")
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "10_arrival_scatter.png", dpi=160)
plt.show()

## Step 4: Simulate a windowed batcher

This is the classic simpler policy: collect requests for a short wait window, run a batch, then repeat. It helps, but it still creates idle gaps and unfair waiting.

In [ ]:
def simulate_windowed_batching(records, max_batch_size=4, max_wait_ms=12, prefill_tps=6400, decode_tps=72, overhead_ms=20):
    ordered = sorted(records, key=lambda item: item["arrival_ms"])
    rows = []
    batch = []
    batch_id = 0
    clock_ms = 0.0

    def flush(batch, batch_id, clock_ms):
        first_arrival = batch[0]["arrival_ms"]
        ready_ms = max(clock_ms, first_arrival + max_wait_ms)
        max_prompt = max(item["prompt_tokens"] for item in batch)
        max_output = max(item["output_tokens"] for item in batch)
        service_ms = 1000 * max_prompt / prefill_tps + 1000 * max_output / decode_tps + overhead_ms
        finish_ms = ready_ms + service_ms
        flushed = []
        for item in batch:
            latency_ms = finish_ms - item["arrival_ms"]
            flushed.append({
                **item,
                "policy": "windowed_batching",
                "batch_id": batch_id,
                "dispatch_ms": round(ready_ms, 1),
                "latency_ms": round(latency_ms, 1),
                "ttft_ms": round(ready_ms - item["arrival_ms"] + 1000 * max_prompt / prefill_tps + 12, 1),
                "throughput_tps": round(item["output_tokens"] / max(latency_ms / 1000, 0.001), 1),
            })
        return flushed, finish_ms

    for record in ordered:
        if not batch:
            batch = [record]
            continue
        first_arrival = batch[0]["arrival_ms"]
        if len(batch) >= max_batch_size or record["arrival_ms"] - first_arrival > max_wait_ms:
            flushed, clock_ms = flush(batch, batch_id, clock_ms)
            rows.extend(flushed)
            batch_id += 1
            batch = [record]
        else:
            batch.append(record)

    if batch:
        flushed, clock_ms = flush(batch, batch_id, clock_ms)
        rows.extend(flushed)

    return pd.DataFrame(rows)

windowed_df = simulate_windowed_batching(stream_df.to_dict("records"))
print(windowed_df[["request_id", "batch_id", "dispatch_ms", "ttft_ms", "latency_ms"]].to_markdown(index=False))

## Step 5: Summarize the windowed policy

The batcher improves throughput, but short requests can still be trapped behind long requests in the same batch.

In [ ]:
windowed_summary = pd.DataFrame([{
    "policy": "windowed_batching",
    "mean_ttft_ms": round(windowed_df["ttft_ms"].mean(), 1),
    "p95_latency_ms": round(percentile(windowed_df["latency_ms"], 95), 1),
    "mean_throughput_tps": round(windowed_df["throughput_tps"].mean(), 1),
}])
print(windowed_summary.to_markdown(index=False))

## Step 6: Simulate continuous batching

Continuous batching admits new requests whenever capacity opens and continues decoding existing sequences instead of forcing all work into rigid request groups.

In [ ]:
def simulate_continuous_batching(records, max_active=4, prefill_tps=8600, decode_tps=84, scheduler_overhead_ms=2):
    waiting = deque(sorted([{**record} for record in records], key=lambda item: item["arrival_ms"]))
    pending_prefill = deque()
    decoding = []
    completed = []
    events = []
    time_ms = 0.0

    def admit_ready_requests(time_ms):
        while waiting and waiting[0]["arrival_ms"] <= time_ms and len(pending_prefill) + len(decoding) < max_active:
            request = waiting.popleft()
            request["remaining_decode"] = request["output_tokens"]
            request["prefill_done"] = False
            pending_prefill.append(request)
            events.append({"event": "admit", "time_ms": round(time_ms, 1), "request_id": request["request_id"], "active_sequences": len(decoding) + len(pending_prefill)})

    while waiting or pending_prefill or decoding:
        if not pending_prefill and not decoding and waiting:
            time_ms = max(time_ms, float(waiting[0]["arrival_ms"]))
            admit_ready_requests(time_ms)
            continue

        admit_ready_requests(time_ms)

        if pending_prefill:
            request = pending_prefill.popleft()
            prefill_ms = 1000 * request["prompt_tokens"] / prefill_tps
            time_ms += prefill_ms + scheduler_overhead_ms
            request["prefill_done"] = True
            request["ttft_ms"] = round(time_ms - request["arrival_ms"] + max(12.0, 1000 / decode_tps), 1)
            decoding.append(request)
            events.append({"event": "prefill_done", "time_ms": round(time_ms, 1), "request_id": request["request_id"], "active_sequences": len(decoding)})
            admit_ready_requests(time_ms)

        if decoding:
            decode_step_ms = 1000 * len(decoding) / decode_tps
            time_ms += decode_step_ms + scheduler_overhead_ms
            finished_ids = []
            for request in decoding:
                request["remaining_decode"] -= 1
                if request["remaining_decode"] <= 0:
                    latency_ms = time_ms - request["arrival_ms"]
                    completed.append({
                        "request_id": request["request_id"],
                        "arrival_ms": request["arrival_ms"],
                        "prompt_tokens": request["prompt_tokens"],
                        "output_tokens": request["output_tokens"],
                        "tenant": request["tenant"],
                        "policy": "continuous_batching",
                        "ttft_ms": request["ttft_ms"],
                        "latency_ms": round(latency_ms, 1),
                        "throughput_tps": round(request["output_tokens"] / max(latency_ms / 1000, 0.001), 1),
                    })
                    finished_ids.append(request["request_id"])
                    events.append({"event": "finish", "time_ms": round(time_ms, 1), "request_id": request["request_id"], "active_sequences": len(decoding) - 1})
            decoding = [request for request in decoding if request["request_id"] not in finished_ids]

    return pd.DataFrame(completed), pd.DataFrame(events)

continuous_df, continuous_events = simulate_continuous_batching(stream_df.to_dict("records"))
print(continuous_df[["request_id", "ttft_ms", "latency_ms", "throughput_tps"]].to_markdown(index=False))

## Step 7: Compare the policies directly

The important question is whether continuous batching reduces waiting without losing delivered token rate.

In [ ]:
continuous_summary = pd.DataFrame([{
    "policy": "continuous_batching",
    "mean_ttft_ms": round(continuous_df["ttft_ms"].mean(), 1),
    "p95_latency_ms": round(percentile(continuous_df["latency_ms"], 95), 1),
    "mean_throughput_tps": round(continuous_df["throughput_tps"].mean(), 1),
}])

policy_comparison = pd.concat([windowed_summary, continuous_summary], ignore_index=True)
print(policy_comparison.to_markdown(index=False))

## Step 8: Plot the scheduler comparison

This chart makes the trade-off legible for teammates who do not want to read the simulator.

In [ ]:
ax = policy_comparison.plot(x="policy", y=["mean_ttft_ms", "p95_latency_ms"], kind="bar", rot=0, figsize=(8, 4), title="TTFT and latency by policy")
ax.set_ylabel("milliseconds")
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "10_policy_comparison.png", dpi=160)
plt.show()

## Step 9: Inspect the active-sequence timeline

Continuous batching is easier to understand when you look at events instead of only aggregates.

In [ ]:
timeline_df = continuous_events.copy()
timeline_df.to_csv(ARTIFACT_DIR / "continuous_batching_events.csv", index=False)
print(timeline_df.head(18).to_markdown(index=False))

## Step 10: Plot active sequences over time

The point of the runtime is not to make queues disappear. The point is to keep useful work in flight more continuously.

In [ ]:
event_plot_df = timeline_df.groupby("time_ms")["active_sequences"].max().reset_index()
ax = event_plot_df.plot(x="time_ms", y="active_sequences", figsize=(8, 4), color="#54a24b", title="Active sequences in the continuous scheduler")
ax.set_xlabel("scheduler time (ms)")
ax.set_ylabel("active sequences")
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "10_active_sequences.png", dpi=160)
plt.show()

## Step 11: Build a paged or managed KV-cache model

Now we switch from scheduling to memory. The managed cache allocates reusable token blocks instead of one fixed contiguous reservation per request.

In [ ]:
def build_allocator_summary(records, block_tokens=128, safety_headroom_tokens=256):
    rows = []
    for record in records:
        peak_tokens = record["prompt_tokens"] + record["output_tokens"]
        contiguous_reserved_tokens = peak_tokens + safety_headroom_tokens
        paged_reserved_tokens = math.ceil(peak_tokens / block_tokens) * block_tokens
        rows.append({
            "request_id": record["request_id"],
            "peak_tokens": peak_tokens,
            "contiguous_reserved_tokens": contiguous_reserved_tokens,
            "paged_reserved_tokens": paged_reserved_tokens,
            "saved_tokens": contiguous_reserved_tokens - paged_reserved_tokens,
            "internal_fragmentation_tokens": paged_reserved_tokens - peak_tokens,
        })
    return pd.DataFrame(rows)

allocator_df = build_allocator_summary(stream_df.to_dict("records"))
print(allocator_df.to_markdown(index=False))

## Step 12: Summarize allocator waste

The managed cache still has block-level internal fragmentation, but it avoids the larger safety reservation that a naive contiguous strategy often needs.

In [ ]:
allocator_summary = pd.DataFrame([{
    "contiguous_reserved_tokens": int(allocator_df["contiguous_reserved_tokens"].sum()),
    "paged_reserved_tokens": int(allocator_df["paged_reserved_tokens"].sum()),
    "saved_tokens": int(allocator_df["saved_tokens"].sum()),
    "fragmentation_tokens": int(allocator_df["internal_fragmentation_tokens"].sum()),
}])
print(allocator_summary.to_markdown(index=False))

## Step 13: Simulate peak block usage over the schedule

A real runtime allocates and frees blocks as requests start and finish. We can approximate that behavior by replaying the scheduler results.

In [ ]:
def replay_block_usage(completed_df, block_tokens=128):
    events = []
    for row in completed_df.to_dict("records"):
        block_count = math.ceil((row["prompt_tokens"] + row["output_tokens"]) / block_tokens)
        start_ms = row["arrival_ms"] + row["ttft_ms"] - 12
        end_ms = row["arrival_ms"] + row["latency_ms"]
        events.append({"time_ms": round(start_ms, 1), "delta_blocks": block_count, "request_id": row["request_id"], "event": "alloc"})
        events.append({"time_ms": round(end_ms, 1), "delta_blocks": -block_count, "request_id": row["request_id"], "event": "free"})
    events = sorted(events, key=lambda item: (item["time_ms"], 0 if item["event"] == "alloc" else 1))
    active_blocks = 0
    timeline = []
    for event in events:
        active_blocks += event["delta_blocks"]
        timeline.append({**event, "active_blocks": active_blocks})
    return pd.DataFrame(timeline)

block_timeline_df = replay_block_usage(continuous_df)
block_timeline_df.to_csv(ARTIFACT_DIR / "block_timeline.csv", index=False)
print(block_timeline_df.head(16).to_markdown(index=False))

## Step 14: Plot block pressure

This is a compact way to reason about memory pressure under concurrency.

In [ ]:
ax = block_timeline_df.plot(x="time_ms", y="active_blocks", figsize=(8, 4), color="#e45756", title="Managed KV-cache block pressure")
ax.set_xlabel("scheduler time (ms)")
ax.set_ylabel("active blocks")
plt.tight_layout()
plt.savefig(ARTIFACT_DIR / "10_block_pressure.png", dpi=160)
plt.show()

## Step 15: Sweep block size choices

Small blocks reduce internal fragmentation. Larger blocks reduce block-table overhead. There is no universal best size.

In [ ]:
block_sweep_rows = []
for block_tokens in [32, 64, 128, 256]:
    sweep_df = build_allocator_summary(stream_df.to_dict("records"), block_tokens=block_tokens)
    block_sweep_rows.append({
        "block_tokens": block_tokens,
        "saved_tokens": int(sweep_df["saved_tokens"].sum()),
        "fragmentation_tokens": int(sweep_df["internal_fragmentation_tokens"].sum()),
        "peak_request_blocks": int(np.ceil(sweep_df["paged_reserved_tokens"].max() / block_tokens)),
    })

block_sweep_df = pd.DataFrame(block_sweep_rows)
print(block_sweep_df.to_markdown(index=False))

## Step 16: Sweep scheduler capacity too

Scheduler and cache decisions interact. More active sequences can improve delivered throughput, but the managed cache footprint rises as well.

In [ ]:
capacity_rows = []
for max_active in [2, 4, 6]:
    cap_df, _ = simulate_continuous_batching(stream_df.to_dict("records"), max_active=max_active)
    capacity_rows.append({
        "max_active": max_active,
        "mean_ttft_ms": round(cap_df["ttft_ms"].mean(), 1),
        "p95_latency_ms": round(percentile(cap_df["latency_ms"], 95), 1),
        "mean_throughput_tps": round(cap_df["throughput_tps"].mean(), 1),
    })

capacity_df = pd.DataFrame(capacity_rows)
print(capacity_df.to_markdown(index=False))

## Step 17: Write down operating rules

By this point the runtime behavior is visible enough to turn into simple deployment heuristics.

In [ ]:
operating_rules = pd.DataFrame([
    {"rule": "Prefer continuous batching when arrival gaps are small", "reason": "It reduces queue buildup without waiting for full rigid batches."},
    {"rule": "Track TTFT and not only total latency", "reason": "Users feel slow first-token delivery immediately."},
    {"rule": "Treat block size as a tuning knob", "reason": "It balances internal fragmentation against allocator metadata overhead."},
    {"rule": "Cap active sequences before memory collapse", "reason": "More concurrency only helps if the KV cache still fits comfortably."},
])
print(operating_rules.to_markdown(index=False))

## Step 18: Export a runtime brief

This brief is the handoff artifact for later notebooks on prefix caching, chunked prefill, and long-context serving.

In [ ]:
runtime_brief = {
    "notebook": "10_continuous_batching_and_paged_kv_cache",
    "policy_comparison": policy_comparison.to_dict("records"),
    "allocator_summary": allocator_summary.to_dict("records"),
    "block_sweep": block_sweep_df.to_dict("records"),
    "capacity_sweep": capacity_df.to_dict("records"),
    "operating_rules": operating_rules.to_dict("records"),
    "artifacts": [
        "10_arrival_scatter.png",
        "10_policy_comparison.png",
        "10_active_sequences.png",
        "10_block_pressure.png",
    ],
}

write_json(ARTIFACT_DIR / "runtime_brief.json", runtime_brief)
print(json.dumps(runtime_brief, indent=2))

## Recap

You now have the two core mechanics behind modern high-throughput runtimes: a scheduler that keeps injecting useful work and a managed KV cache that avoids brittle monolithic reservations. The next notebook builds on this with prefix caching, chunked prefill, and long-context behavior.